In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
from crmprtd import Row
import logging
import os
import pickle
import pandas as pd
import numpy as np
from natsort import natsorted
from natsort import natsort_keygen
from pprint import pprint
from collections import Counter
from collections import defaultdict

import re
from rapidfuzz import fuzz
from crmprtd import infer
from fern_func import *
# from sql_func import *
import sqlalchemy as sa
from fern_raw_db_dompare import *

In [10]:
engine = sa.create_engine("postgresql://tongli1997@db.pcic.uvic.ca:5433/crmp?keepalives=1&keepalives_idle=300&keepalives_interval=300&keepalives_count=9&passfile=/workspaces/crmprtd/.pgpass", echo=False)
session = Session(engine)

In [7]:
station_native_id_1 = 'Dennis'
station_native_id_2 = 'McDonn'


# Your 'McDonn' data has the correct start date, but the end date is Jul 6, 2022, please delete data after that. For the 'Dennis' station, please delete the data before Jul 6, 2022 as you are correct it is duplicate data. 

In [8]:
from sqlalchemy import text, create_engine


sql_dennis = text("""
DELETE FROM obs_raw o
USING meta_history h, meta_station s
WHERE o.history_id = h.history_id
  AND h.station_id = s.station_id
  AND s.network_id = 11
  AND s.native_id = :station
  AND o.obs_time < :cutoff
""")

sql_mcdonn = text("""
DELETE FROM obs_raw o
USING meta_history h, meta_station s
WHERE o.history_id = h.history_id
  AND h.station_id = s.station_id
  AND s.network_id = 11
  AND s.native_id = :station
  AND o.obs_time > :cutoff
""")

with engine.begin() as conn:  # auto-commit transaction
    conn.execute(sql_dennis, {"station": "Dennis", "cutoff": "2022-07-06"})
    conn.execute(sql_mcdonn, {"station": "McDonn", "cutoff": "2022-07-06"})

## Validate after the del commend

In [11]:

sql_times = text("""
SELECT s.native_id, MIN(o.obs_time) AS earliest, MAX(o.obs_time) AS latest
FROM obs_raw o
JOIN meta_history h ON o.history_id = h.history_id
JOIN meta_station s ON h.station_id = s.station_id
WHERE s.native_id IN (:station1, :station2)
  AND s.network_id = 11
GROUP BY s.native_id
""")

with engine.connect() as conn:
    df_times = pd.read_sql(sql_times, conn,
                           params={"station1": "Dennis", "station2": "McDonn"})

print(df_times)

  native_id            earliest              latest
0    Dennis 2022-07-06 00:00:00 2025-09-23 10:00:00
1    McDonn 2021-07-07 17:00:00 2022-07-06 00:00:00
